In [ ]:
# =============================================================================
# MEMORY-LIGHT VALIDATOR — same rules and messages as utils/validate_submission.py
# =============================================================================
# Why this exists: the official validator stores every candidate list in memory
# (only to run its soft "matches are a subset of candidates" warning). With
# ~252M candidate IDs that exceeds Kaggle's RAM and the process is killed
# (exit code -9, no output).
#
# This version applies the SAME rules, in the same order, with the same error
# and warning wording, but streams candidate_pairs.tsv one line at a time and
# runs the subset check on the fly. Peak memory ~2 GB even with CHECK_IDS on.
#
# Standalone: needs nothing from earlier cells. Runtime ~5-10 min on 3.3 GB.
# =============================================================================

import os
import time
from pathlib import Path
from tqdm.auto import tqdm

DATA_ROOT = Path("/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/"
                 "student_resource/dataset")
WORK = Path("/kaggle/working")

TEST_DIR = DATA_ROOT / "test"
CANDIDATE = WORK / "candidate_pairs.tsv"
MATCHING = None      # None -> a placeholder with empty matches is generated,
                     # so candidate_pairs.tsv can be checked on its own.
                     # Later: WORK / "matching_results.tsv"
CHECK_IDS = True     # also verify every ID exists in test_source2/3 (~1.5 GB RAM)

DELIM = "\t"
MAX_EXAMPLES = 5
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    """First-column IDs of a source TSV (header skipped, blank lines ignored)."""
    with open(path, encoding="utf-8") as f:
        next(f, None)
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    """Short, human-readable sample of offending IDs (same as the official)."""
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def validate_file(path, expected_header, col_label, required, valid_ids, errors,
                  keep_mapping, matched=None, subset_offenders=None):
    """
    Official per-file rules, streamed.
      keep_mapping=True  -> return {s1: set(ids)} (used for the small matching file)
      keep_mapping=False -> nothing stored per row; if `matched` is given, the
                            subset check runs on each row as it is read.
    Returns (mapping or {}, set of S1 rows seen) or (None, None) on a fatal error.
    """
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None, None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None, None
        if DELIM not in header and "," in header:
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv); "
                "write it with df.to_csv(sep='\\t', index=False)."
            )
            return None, None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None, None

        lines = tqdm(f, total=len(required), desc=f"checking {name}", unit="row",
                     unit_scale=True, mininterval=10)
        for line_num, line in enumerate(lines, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                if keep_mapping:
                    mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            if keep_mapping:
                mapping[s1] = id_set
            if matched is not None:
                mids = matched.get(s1)
                if mids and mids - id_set:
                    subset_offenders.add(s1)
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    findings = [
        (dup_rows, "{name}: duplicate source1_entity_id row(s): {ex}. "
                   "Each S1 entity may appear on only one row."),
        (intra_dupes, "{name}: repeated ID inside a {col} list for: {ex}. "
                      "No duplicate IDs are allowed within a list."),
        (self_matches, "{name}: {col} contains Source-1 IDs (self-matches): {ex}. "
                       "Only S2-/S3- IDs are allowed."),
        (wrong_prefix, "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}."),
        (unknown, "{name}: {col} references IDs not in the test "
                  "Source-2/3 files: {ex}."),
        (required - seen, "{name}: required S1 entity(ies) missing: {ex}. "
                          "Every entity in test_source1.tsv needs a row (empty = no match)."),
        (seen - required, "{name}: row(s) using an S1 ID that is not in the test set: {ex}."),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping, seen


def validate(matching_path, candidate_path, test_dir, check_ids):
    """Same flow as the official validate(); returns (errors, warnings)."""
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    valid_ids = None
    if check_ids:
        valid_ids = set()
        for nm in ("test_source2.tsv", "test_source3.tsv"):
            p = os.path.join(test_dir, nm)
            if not os.path.isfile(p):
                warnings.append(f"{p} not found — skipping the (optional) check that "
                                "matched IDs exist in the test set.")
                valid_ids = None
                break
            valid_ids |= read_ids(p)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        warnings.append("ID-existence check is OFF — not checking that matched/"
                        "candidate IDs exist in the test set.")

    matched, _ = validate_file(matching_path, MATCHING_HEADER, "matched_entity_ids",
                               required, valid_ids, errors, keep_mapping=True)

    if candidate_path and os.path.isfile(candidate_path):
        offenders = set()
        _, cand_seen = validate_file(candidate_path, CANDIDATE_HEADER,
                                     "candidate_entity_ids", required, valid_ids, errors,
                                     keep_mapping=False, matched=matched,
                                     subset_offenders=offenders)
        if matched is not None and cand_seen is not None:
            # rows with matches but no candidate row at all also count (official behaviour)
            offenders |= {s1 for s1, mids in matched.items() if mids and s1 not in cand_seen}
            if offenders:
                warnings.append(
                    f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                    f"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches "
                    "normally come from your blocking candidates — double-check these."
                )
    elif candidate_path:
        warnings.append(f"{candidate_path} not found — skipping candidate_pairs.tsv checks.")

    return errors, warnings


def make_placeholder_matching(test_dir, out_path):
    """Matching file with every test S1 row and no matches, so the candidate file
    can be validated before a real matching file exists."""
    with open(os.path.join(test_dir, "test_source1.tsv"), encoding="utf-8") as src, \
         open(out_path, "w", encoding="utf-8") as out:
        next(src, None)
        out.write("source1_entity_id\tmatched_entity_ids\n")
        for line in src:
            if line.strip():
                out.write(line.split(DELIM, 1)[0].strip() + "\t\n")


def run(matching, candidate, test_dir, check_ids):
    """Print a report like the official validator and return its exit code."""
    t = time.time()
    placeholder = None
    if matching is None:
        placeholder = Path(WORK) / "_placeholder_matching.tsv"
        make_placeholder_matching(test_dir, placeholder)
        matching = placeholder
        print("  (no matching file given: using a placeholder with empty matches)")

    print("ML Challenge 2026 — memory-light validator (official rules)")
    print(f"  test dir: {test_dir}")
    try:
        errors, warnings = validate(str(matching), str(candidate) if candidate else None,
                                    str(test_dir), check_ids)
    except UnicodeDecodeError:
        print("\nFAIL — 1 issue(s) to fix before submitting:")
        print("  1. A file is not valid UTF-8 text.")
        return 1
    finally:
        if placeholder is not None and placeholder.exists():
            placeholder.unlink()

    print()
    for w in warnings:
        print(f"WARNING: {w}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, e in enumerate(errors, 1):
            print(f"  {i}. {e}")
        code = 1
    else:
        print("PASS — no blocking issues found. Safe to submit.")
        code = 0
    print(f"  ({(time.time()-t)/60:.1f} min)")
    return code


if __name__ == "__main__":
    exit_code = run(MATCHING, CANDIDATE, TEST_DIR, CHECK_IDS)